# Module 4 — The FUR Regulon: Literature to Motif

**Lab context:** *E. coli* K-12 | bowtie2 | MEME Suite | Biopython | FUR transcription factor

---

This module connects a published dataset to a motif analysis pipeline:

1. **Understanding the literature** — read a real ChIP-exo paper and extract biological context.
2. **RNA-seq paired-end alignment** — the foundational read-alignment step (extends Module 3), applied to the study's transcriptome data.
3. **Motif discovery with MEME** — given a set of genomic sequences, MEME finds overrepresented patterns.
4. **Sequence extraction with Biopython** — pull binding-site sequences from the *E. coli* K-12 genome by coordinate.
5. **TSS distance analysis** — compute distances between binding sites and transcription start sites.

---

**Learning objectives:**
- Understand a real ChIP-exo study and the data it produced
- Use Biopython to extract genomic sequences by coordinate from a public dataset
- Run MEME and interpret motif output (E-value, logo shape)
- Download and align **paired-end RNA-seq** reads with bowtie2, and turn them into a GFF track
- Compute TSS distances as a downstream regulatory analysis


## Background: Transcription Factors, FUR, and Motifs

### What is a transcription factor?

A **transcription factor (TF)** is a protein that binds to specific short DNA sequences near a gene and controls whether that gene gets expressed. Think of it as a light switch wired into the chromosome: when the TF binds, it turns the gene on (or off). Bacteria like *E. coli* use hundreds of TFs to respond to changing conditions — temperature, nutrients, stress.

### What is FUR?

**FUR** (Ferric Uptake Regulator) is one of the most-studied TFs in *E. coli*. Its job is to sense how much iron is available in the cell and adjust gene expression accordingly:

- **Iron-replete (plenty of iron):** FUR binds its target sites and *represses* iron-uptake genes (no need to import more iron)
- **Iron-depleted (low iron):** FUR releases its targets, allowing iron-uptake genes to be expressed

Iron is essential for most cellular processes but toxic in excess — FUR is the cell's main iron thermostat. It controls roughly 90 genes in *E. coli*.

### What is a regulon?

A **regulon** is the complete set of genes controlled by one TF. The FUR regulon = all genes whose expression is directly regulated by FUR binding. Mapping a regulon requires knowing *where* FUR binds — which is exactly what ChIP-exo (Module 3) measures.

### What is a DNA binding motif?

When a TF binds DNA, it doesn't land randomly — it recognizes a specific short sequence pattern, typically 15–20 bp. This pattern is the TF's **binding motif**. For FUR it's called the **Fur box**. If you look at all the genomic sites where FUR binds, you expect to find the same short sequence pattern repeated at each one.

**MEME** is the tool that discovers this pattern automatically: it takes a set of sequences (your ChIP-exo peaks) and finds the motif that appears more often than chance would predict.

---

## 1. The FUR Regulon — Seo et al. 2014

The FUR binding site data used in this module comes from a real published study:

> **Seo SW, Kim D, Latif H, O'Brien EJ, Szubin R, Palsson BO (2014)** — *Deciphering Fur transcriptional regulatory network highlights its complex role beyond iron metabolism in Escherichia coli* — ***Nature Communications*** 5:4910.
> PMID: 25222563 | PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC4167408/ | DOI: 10.1038/ncomms5910 | GEO: **GSE54901**

The study used ChIP-exo to map Fur-binding sites genome-wide across different iron conditions.

**Two different things live in two different places — this matters for the exercises:**
- **Raw ChIP-exo signal** is deposited at GEO (**GSE54901**), as per-base *coverage tracks* (one row per genomic position). This is what you align to and visualize — it is **not** a list of binding sites.
- **The discrete binding-site coordinates** (the 144 called peaks, with their target genes and signal strengths) are in the paper's **Supplementary Data** table (an Excel file), *not* on GEO. This table is the input to your MEME analysis in Exercise 5.


### Exercise 1 — Understand the paper and its data

Read Seo et al. 2014 (PMC4167408) — or use Claude Code to help you understand it — and write a **3-sentence summary** in the cell below covering:
- What experiment was done, in what organism, and under what conditions
- How many FUR binding sites were identified and what regulatory modes (categories) were described
- Where the discrete binding-site coordinates are found (which supplementary table), and — separately — what kind of data GEO GSE54901 actually holds

In Exercise 5 you will download the **supplementary binding-site table** from the paper and use its coordinates as the input to motif discovery.


> **Answer**
>
> *Your 3-sentence summary here.*
> 
> 실험이 어떤 개체 내에서 어떤 환경에서 어떤 작업을 수행하였는지 / FUR binding site가 몇개가 발견되었고 조절 방법(카테고리?)가 설명되었는지 / 어던 보충 자료(테이블)에서 bingind site가 설명되었는지(이거 다시 해석), GEO GSE54901이 실제로 무엇을 잡고있는지
>
>
> 1. what to do : FUR 조절 네트워크 규명이 목적 -> 이를 위해 FUR과 RNA polymerase를 대상으로 chip-exo를 하여 얻은 데이터와 cDNA 시퀀싱 데이터를 이용. 이 과정에서 FUR regulon을 재구성하기 위해 결합 부위 확인과 genome scale에서 mRNA transcription level을 확인. -> 이 데이터들을 기반하여 apo, holo FUR에 의한 서로다른 활성화 기작과 억제 기작을 규명함.    실험 균주로는 *E.coli* K-12 MG1655가 이용됨.
> 
> 2. FUR의 binding site는 총 iron starvation cond에서 59개, iron replet cond에선 118개가 나왔다. starvation cond에서 나온 59개 중 58개는 replet cond와 같은 위치에서 확인되었다. Different mode of FUR regulation -> apo- & holo- FUR 활성화하고, holo- FUR 억제한다.
>
>
> 3. Binding site의 개별 좌표(coordinate)는 Fig. 2a와 Supplementary Data 1에 나타나있다. GEO GSE54901은 Fur/RNAP ChIP-exo와 RNA-seq의 raw data를 담고 있음
> 


## 2. RNA-seq: Paired-End Alignment with `bowtie2`

So far you've worked with **ChIP-exo** data (single-end). The same Fur study also produced **RNA-seq** data — a *second data type* and your first **paired-end** alignment.

Paired-end sequencing reads **both ends** of each DNA fragment, producing two FASTQ files per sample (`_1` = R1, `_2` = R2). `bowtie2` aligns them together (`-1`/`-2`), using the known fragment-size range to place reads more accurately than single-end. In this section you'll download an RNA-seq sample, align it paired-end, and turn it into a GFF — the same pipeline shape as Module 3, but for a new data type.

**Data:** the RNA-seq sub-series of the Fur study is **GEO GSE54900** (same paper, *Nature Communications* 5:4910). In the original lab workflow the paired-end sample `SRR1168133` was aligned with `bowtie2 --very-fast -X 1000 -3 3` (paired-end), then sorted and converted to GFF with `makegff.py`.


### Exercise 2 — Run a paired-end RNA-seq alignment

**Step 1 — Get a paired-end RNA-seq sample.**
Use Claude Code to navigate to the RNA-seq series and pick one paired-end sample:
```
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE54900
```
Ask Claude Code to help you:
1. Find a **paired-end** RNA-seq run in GSE54900 and get its SRR accession.
2. Download it with `fastq-dump --split-files <SRR>` (or `fasterq-dump`) — paired-end gives you **two** files, `<SRR>_1.fastq` and `<SRR>_2.fastq`.

**Step 2 — Align paired-end.**
Reuse the *E. coli* K-12 bowtie2 index you built in Module 3. Run a **paired-end** alignment — note the `-1`/`-2` inputs (not a single `-U`):
```bash
bowtie2 --very-fast -X 1000 -3 3 -p 2 \
    -x <index_prefix> \
    -1 <SRR>_1.fastq -2 <SRR>_2.fastq \
    -S <SRR>.sam
```
Then `samtools view -bS` → `samtools sort` → `samtools index`, exactly as in Module 3.

**Step 3 — Make a GFF for the RNA-seq data.**
Run your `makegff.py` from Module 3 on the sorted BAM to produce a coverage GFF (this is RNA-seq signal, so both strands are meaningful):
```bash
python makegff.py --separate_strand <SRR>.sorted.bam rnaseq.gff
```
This GFF can be loaded into **MetaScope** alongside the gene annotation to visualize transcription — the same way you visualize the ChIP-exo track.

**Step 4 — Explain the flags.** In the Answer cell, write a **one-sentence explanation for each** flag below (focus on *why*, not just mechanics), drawing on what you saw when you ran the alignment:
- `-X 1000` (maximum fragment/insert size — a paired-end-only flag)
- `-3 3` (3′ trimming)
- `--no-mixed` and `--no-discordant` (paired-end concordance)

Also answer: **why is `-X` meaningless for the single-end ChIP-exo alignment in Module 3, but important here?** And: what would go wrong downstream if you forgot `--no-discordant`?

> **Single-end vs paired-end:** Module 3 aligned ChIP-exo reads with a single `-U` input. Here you use `-1`/`-2`. If you accidentally pass paired files to `-U`, bowtie2 treats them as unpaired and every `-X`/concordance flag is silently ignored — a common and quiet mistake.

Write your working pipeline in the code cell below (or run it in the terminal and paste the commands here).


In [3]:
%%bash
# Paired-end RNA-seq pipeline (GSE54900) — fill in the commands after generating them with
# Claude Code. This is a bash cell: the FIRST line MUST be %%bash, or Jupyter will try to
# run these shell commands as Python and fail with a SyntaxError.
# Note: this notebook's shell cwd is notebooks/, so repo-root paths need a ../ prefix
# (same convention as Module 3). prefetch/fastq-dump are installed at /opt/sratoolkit/bin,
# which is not on PATH, so they're called with their full path below.

# Step 1: Download the paired-end RNA-seq sample (fastq-dump --split-files -> _1.fastq / _2.fastq)
# GSE54900 -> SRP037711 -> SRX469838 (WT with Fe, replicate 1) -> SRR1168133 (paired-end, 10,848,313 spots)
/opt/sratoolkit/bin/prefetch SRR1168133 -O ../data/reference/
/opt/sratoolkit/bin/fastq-dump --split-files --gzip -O ../data/reference/ ../data/reference/SRR1168133/SRR1168133.sra
rm -rf ../data/reference/SRR1168133

# Step 2: Align paired-end reads, reusing the Module 3 bowtie2 index (-1/-2, not -U)
bowtie2 --very-fast -X 1000 -3 3 -p 2 \
    -x ../data/reference/NC_000913.3 \
    -1 ../data/reference/SRR1168133_1.fastq.gz -2 ../data/reference/SRR1168133_2.fastq.gz \
    -S ../data/reference/SRR1168133.sam
# overall alignment rate: 98.31%

# Step 3: SAM -> sorted, indexed BAM (samtools view / sort / index)
samtools view -bS ../data/reference/SRR1168133.sam > ../data/reference/SRR1168133.bam
samtools sort ../data/reference/SRR1168133.bam -o ../data/reference/SRR1168133_sorted.bam
samtools index ../data/reference/SRR1168133_sorted.bam

# Step 4: Run makegff.py on the sorted BAM to make the RNA-seq coverage GFF
# Uses the repo-root makegff.py (argparse-based, supports --separate_strand);
# notebooks/makegff.py is the Module 3 ChIP-exo-only variant and doesn't have that flag.
python ../makegff.py --separate_strand ../data/reference/SRR1168133_sorted.bam ../data/reference/rnaseq.gff


2026-07-21T08:44:52 prefetch.3.2.1: 1) Resolving 'SRR1168133'...
2026-07-21T08:44:54 prefetch.3.2.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-07-21T08:44:55 prefetch.3.2.1: 1) Downloading 'SRR1168133'...
2026-07-21T08:44:55 prefetch.3.2.1:  SRA Normalized Format file is being retrieved
2026-07-21T08:44:55 prefetch.3.2.1:  Downloading via HTTPS...
2026-07-21T08:45:24 prefetch.3.2.1:  HTTPS download succeed
2026-07-21T08:45:25 prefetch.3.2.1:  'SRR1168133' is valid: 391400687 bytes were streamed from 391391479
2026-07-21T08:45:25 prefetch.3.2.1: 1) 'SRR1168133' was downloaded successfully
2026-07-21T08:45:25 prefetch.3.2.1: 1) Resolving 'SRR1168133's dependencies...
2026-07-21T08:45:25 prefetch.3.2.1: 'SRR1168133' has 0 unresolved dependencies
Read 10848313 spots for ../data/reference/SRR1168133/SRR1168133.sra
Written 10848313 spots for ../data/reference/SRR1168133/SRR1168133.sra


[WARNING] Failed to launch x86-64-v3 version, staying with default
[WARNING] Failed to launch x86-64-v3 version, staying with default
10848313 reads; of these:
  10848313 (100.00%) were paired; of these:
    640168 (5.90%) aligned concordantly 0 times
    9891156 (91.18%) aligned concordantly exactly 1 time
    316989 (2.92%) aligned concordantly >1 times
    ----
    640168 pairs aligned concordantly 0 times; of these:
      244675 (38.22%) aligned discordantly 1 time
    ----
    395493 pairs aligned 0 times concordantly or discordantly; of these:
      790986 mates make up the pairs; of these:
        367530 (46.46%) aligned 0 times
        339787 (42.96%) aligned exactly 1 time
        83669 (10.58%) aligned >1 times
98.31% overall alignment rate
[bam_sort_core] merging from 5 files and 1 in-memory blocks...


> **Answer**
>
> **`-X 1000`:** 
> : paired end는 DNA fragment 양 끝 R1/R2에서 읽은거라, genome에 매핑된 두 위치 사이의 거리가 곧 원래 DNA fragment의 size를 의미하는데, -X는 이 size에 제한을 둬서 size가 너무 큰 경우 fragment로 인지하지 않도록 설정한다.
>
> **`-3 3`:** *Your one-sentence explanation.*
>
> **`--no-mixed` / `--no-discordant`:** *Your one-sentence explanation.*
>
> **Why `-X` matters here but not in Module 3 (single-end):** module3에선 single-end read파일을 지정하는 플래그인 -U를 이용해서 정렬하였으므로 fragment로 볼 수 있는 부분이 없어서 -X를 쓰는 것이 무의미하다.
>
> **What goes wrong without `--no-discordant`:** *Your answer.*


## 3. How MEME Works

### What problem does MEME solve?

You have 90 genomic sequences — one around each FUR binding site. If FUR binds a specific sequence pattern (the Fur box), that pattern should appear in most of your sequences. But you don't know in advance *what* the pattern looks like or exactly *where* in each sequence it sits.

**MEME** (Multiple Em for Motif Elicitation) finds overrepresented sequence patterns in a set of input sequences without knowing what to look for. It uses an algorithm called Expectation-Maximization (EM) to iteratively refine a motif model until it fits the data as well as possible.

### What is an E-value?

The **E-value** answers: *"How many times would I expect to find a motif this good by random chance in a dataset this size?"*

- E-value of 10⁻²⁰ → extremely unlikely by chance → strong, real motif
- E-value of 1.5 → you'd expect to see this pattern 1–2 times by chance → likely noise

For a real TF binding motif like the Fur box, expect E-values well below 10⁻¹⁰. If your top motif has an E-value above 0.01, something may be wrong with your input sequences (wrong coordinates, wrong organism, mixed conditions).

Use Claude Code to understand how the EM algorithm works and what the E-value measures before running MEME.

### Exercise 3 — Understand MEME in your own words

Use Claude Code to explore the MEME algorithm and E-value interpretation. Then write a **3-sentence summary in your own words** — not copied from Claude's output, but re-stated in your own understanding.

> Focus on: what problem EM is solving, what E-value measures, and why motif width matters.

> **Answer**
>
> *Your 3-sentence summary here.*


### Exercise 4 — Conceptual prediction (answer before asking Claude)

**Without asking Claude Code first:** predict what the MEME output would look like for each of these two inputs:

(a) 10 **random** 200 bp genomic sequences from *E. coli* K-12  
(b) 10 **real FUR binding site** sequences (±50 bp around known sites)

For each case, predict: approximate E-value range, whether a clear motif logo would appear, and what width you would expect.

Write your prediction below. Then verify your reasoning with Claude Code and note where you were right or wrong.

*Your prediction here (before checking with Claude Code).*

---

*Verification notes (after checking with Claude Code):*

## 4. Extracting Sequences with Biopython

The Fur-binding site coordinates come from the **Supplementary Data table of Seo et al. 2014** (an Excel file you download from the paper in Exercise 5), which lists each site's `ChIP-exo Start` / `ChIP-exo End`. Use Biopython to extract DNA sequences from the *E. coli* K-12 reference FASTA at those coordinates. Handle strand correctly — reverse-complement sequences on the minus strand.


### Exercise 5 — Fetch the data and write the sequence extraction script

**Step 1 — Get the binding-site table with Claude Code:**
The discrete Fur binding sites are in the **Supplementary Data** of the Nature Communications paper, not on GEO. Give Claude Code the article page and ask it to locate and download the supplementary table:

```
https://www.nature.com/articles/ncomms5910
```

Ask Claude Code to:
1. Find the **Supplementary Data** file that contains the Fur ChIP-exo **binding sites** (target gene, `ChIP-exo Start`, `ChIP-exo End`, binding condition, S/N ratio) and download it into `data/reference/`. *(It is an Excel `.xlsx` file — the same one used in the original lab tutorial.)*
2. Download the *E. coli* K-12 reference FASTA into `data/reference/NC_000913.fasta` (same file as Module 3 — skip if already present).

**Step 2 — Read the table.** It has a title row above the real header, so read it with the header on the **second** row:
```python
import pandas as pd
df = pd.read_excel("data/reference/<supplementary_file>.xlsx", header=1)
df = df.dropna(subset=["Peak"])          # keep rows that are actual binding sites
```
Columns include: `Transcription Unit`, `Peak` (a peak **ID label** like `P1`, `P2` — *not* a score), `ChIP-exo Start`, `ChIP-exo End`, `Binding Condition` (`R` = iron-replete, `S` = iron-depleted, `R/S` = both), `S/N ratio` and `Significance (p-value)` (these are the strength/score fields), and `Distance to TSS`. Filter to the **iron-replete** sites with `Binding Condition` containing `R`.

**Step 3 — Write a Biopython script that:**
1. Reads the binding-site coordinates from the table (iron-replete sites).
2. Loads the reference FASTA.
3. Extracts sequences ±50 bp around each binding site center (center = midpoint of `ChIP-exo Start`/`End`).
4. Handles strand correctly if a strand is available (reverse complement for `-`).
5. Writes a FASTA file (`data/reference/fur_sites_for_meme.fasta`) ready for MEME input.

> **Coordinate convention warning:** the table's coordinates are **1-based**. Python string slicing is **0-based**. Off-by-one errors here silently shift every extracted sequence.

> **Genome-version note:** the paper's coordinates were called on the *E. coli* K-12 MG1655 genome. Extract from a reference FASTA consistent with those coordinates, and sanity-check a couple of sequences (e.g. does a known Fur site contain a recognizable Fur box?) before trusting all of them.

Run the script and confirm the output FASTA has about one sequence per binding site before proceeding. For reference, the table has **144** sites total; **143** are bound under the iron-replete condition (82 labelled `R` = iron-replete only, plus 61 labelled `R/S` = bound in both conditions), and only 1 is `S`-only.

Write the final working script in the code cell below.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# Your Biopython sequence extraction script here.


## 5. Running MEME

MEME is a command-line tool. The basic invocation is:

```bash
meme input.fasta -dna -oc output_dir -mod zoops -minw 15 -maxw 25 -nmotifs 3
```

Use **Claude Code plan mode** to generate the full command with the flags appropriate for your FUR binding site dataset. Make sure you understand each flag before running.

> **Think first:** FUR binds double-stranded DNA, so a binding motif can appear on either strand. Which MEME flag accounts for this? Write your answer before generating the command.

In [ ]:
%%bash
# Run MEME on your extracted sequences.
# Use Claude Code with plan mode to generate the full meme command with appropriate flags.
# Annotate each flag with a comment explaining what it does.

# meme [input.fasta] [flags]


### Exercise 6 — Interpret your MEME results

After running MEME:
1. Use Claude Code to help you read the `meme.txt` or `meme.html` output.
2. Then write your **own assessment** — do not just copy Claude's interpretation.

Address all three of the following:
- **E-value:** Is the top motif statistically significant? What does the E-value tell you?
- **Motif logo:** Does the shape match what you expect for a FUR binding site (the "Fur box")? Which positions are most conserved?
- **Biological meaning:** Given your background from Exercise 1 (the published FUR regulon), is this result consistent with the literature? Why or why not?

Write your assessment in the markdown cell below.

> **Answer**
>
> *Your MEME result assessment here.*


## 6. TSS Distance Analysis

A common downstream analysis computes the distance between each transcription factor binding site and the nearest transcription start site (TSS).

### Exercise 7 — TSS distance script

Two data sources are available for this analysis:
- **Binding sites:** the Fur binding-site table you downloaded in Exercise 5 (`ChIP-exo Start`/`End` per site). The table also contains a `Distance to TSS` column for a subset of sites — you can use this to **cross-check** your own computation.
- **Gene annotation:** `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (pre-provided — lab *E. coli* K-12 gene annotation; all rows are `gene` features, so a TSS must be derived from each gene's start/strand: gene start for `+`, gene end for `-`)

Use Claude Code to generate a Python script that:
1. Reads the binding-site coordinates and the annotation GFF.
2. For each Fur binding site, computes the distance to the nearest TSS.

> **Coordinate-space warning:** the binding-site table has no chromosome column — its coordinates are positions on the *E. coli* K-12 genome. The annotation GFF uses seqname `NC_000913`, while the reference FASTA you downloaded is `NC_000913.3`. When you compare binding-site positions to gene positions, make sure both are in the **same coordinate space** (same genome build) — otherwise distances will be systematically off, silently. This is a good place to run a quick sanity check (does a known site land near its known target gene?).
3. Reports: minimum distance, median distance, and fraction of binding sites within 500 bp of a TSS.
4. (Optional) Plots the distance distribution as a histogram.

Write the final working script in the code cell below.

> **Cross-check:** compare your computed distances against the paper's `Distance to TSS` column for the sites that have one. If they disagree by a constant offset, you likely have an off-by-one or a genome-version mismatch.

> **Biological framing:** Based on Exercise 1 (the Seo et al. 2014 paper), what distance distribution do you expect? Write your prediction as a comment at the top of the script.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# TSS distance analysis script.
# Prediction (write before running): ...



## 7. Integrative Visualization in MetaScope

So far this module produced three separate results: an **RNA-seq coverage** track (Section 2), a set of **Fur binding sites** (the table from Exercise 5), and a **motif** (Section 5). Each answers part of the FUR story, but the story only comes together when you see them **on the same axis**.

That is what MetaScope is for. In Module 3 you visualized a single ChIP-exo track. Here you'll overlay **three tracks** and read the regulation directly off the screen:

| Track | File | What it shows |
|-------|------|---------------|
| Gene annotation | `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` | Where the genes are |
| RNA-seq coverage | `rnaseq.gff` (your Exercise 2 output) | How strongly each region is transcribed |
| Fur binding sites | `fur_sites.gff` (you'll build it below) | Where FUR binds |

The biological payoff: at an iron-uptake gene, you should be able to see a **Fur binding site** sitting just upstream **and** ask whether the gene's **RNA-seq signal** is consistent with FUR's role as a repressor.

> As in Module 3, MetaScope runs on **your own computer**, not in the Codespace. Install it from the lab homepage ([sbml-lab.ai](https://sbml-lab.ai)). Produce the GFFs here, download them, then open them locally.

### Exercise 8 — Turn the Binding Sites into a Track

Your RNA-seq track and the annotation are already GFF files. The binding sites, though, are still rows in an Excel table — MetaScope can't read those. Convert them.

Use Claude Code to write a short script that reads the **iron-replete** binding sites (the `R` / `R/S` rows you filtered in Exercise 5) and writes a GFF file `data/reference/fur_sites.gff`, one row per site:

```
NC_000913	furchip	fur_site	<ChIP-exo Start>	<ChIP-exo End>	<S/N ratio>	.	.	name=<Transcription Unit>
```

Two things that matter (both are recurring themes in this course):
- **Chromosome ID must match the other tracks.** Use `NC_000913` in column 1 so this track lands on the *same* chromosome tab as the annotation. (Recall the Module 3 gotcha: MetaScope groups tracks by column 1.)
- **Put the S/N ratio in the score column** so stronger sites can be told apart when you display the track.

Write the script below, run it, and confirm `fur_sites.gff` has roughly one row per iron-replete site (~143).

In [ ]:
# Build fur_sites.gff from the binding-site table (iron-replete sites).
# Reuse the DataFrame you loaded in Exercise 5 (pd.read_excel(..., header=1)).
# Column 1 MUST be NC_000913 so this track lines up with the annotation in MetaScope.



### Exercise 9 — Overlay Three Tracks and Read the Regulation

On your machine, open MetaScope and load **all three** GFFs (`Ctrl/Cmd+O`): the annotation, `rnaseq.gff`, and `fur_sites.gff`.

1. **Pick a Fur-regulated iron gene.** Use `Ctrl/Cmd+F` to search the annotation, or `Ctrl/Cmd+G` to jump to a coordinate. Good candidates (verified in the lab annotation): **`fepA`** (~609–612 kb), the **`entCEBA`** enterobactin cluster (~624–629 kb), **`fhuA`** (~167 kb), or **`sodB`** (~1,733 kb).
2. **Line up the three tracks.** At your chosen gene, check: is there a **Fur binding site** just upstream? What does the **RNA-seq coverage** look like over the gene body?
3. **Note the RNA-seq condition.** FUR represses iron-uptake genes **when iron is replete**. Whether you expect high or low RNA-seq signal depends on the iron condition of the sample you aligned in Exercise 2 — state which condition it is and reason accordingly. (If you're unsure, that's a `/explain` or a quick check of the GSE54900 sample metadata.)
4. **Adjust the display** so the figure is readable: set distinct track colors (`Ctrl/Cmd+Shift+C`) and heights (`Ctrl/Cmd+Shift+H`), and scale the RNA-seq track so its peaks are visible.
5. **Export** (`Ctrl+Shift+E` → PNG, 300 dpi) as `module4_integrative_metascope.png`.

This exported figure is your Module 4 visualization deliverable.

> **Answer**
>
> - Gene chosen (name / locus_tag): 
> - Is there a Fur binding site upstream of it? Position:
> - RNA-seq condition of your sample (iron-replete / iron-depleted):
> - Is the gene's RNA-seq signal consistent with FUR repression **under that condition**? Explain.
> - Attach `module4_integrative_metascope.png`.

### Exercise 10 — Tie It Back to the Motif and the Literature

You now have three lines of evidence at one locus: a **binding site** (the track), a **motif** (Section 5 — is there a Fur box in that site's sequence?), and an **expression level** (the RNA-seq track). Together these are the core of what Seo et al. 2014 reported.

Write a short interpretation (3–5 sentences):
- Does the binding + expression pattern at your gene match FUR acting as an iron-responsive **repressor**?
- Is this **confirmatory** (a known FUR target), or did you land somewhere **unexpected** (FUR's "role beyond iron metabolism," as the paper's title puts it)?
- Use Claude Code to cross-check the gene's function and known FUR regulation — and note anything that *didn't* line up, and how you resolved it.

This is exactly the kind of integrative reading your mini-project (Module 5) will ask you to do independently.

> **Answer**
>
> *(Your 3–5 sentence integrative interpretation — binding + motif + expression, confirmatory vs unexpected, and what you cross-checked.)*

## End of Module 4

You have completed:
- Reading and understanding a real ChIP-exo paper (Seo et al. 2014)
- Downloading and running a **paired-end RNA-seq** alignment, and producing a GFF track from it
- Using Biopython to pull binding-site sequences from a reference genome
- Running MEME and interpreting motif output
- Computing TSS distances as a downstream regulatory analysis
- **Overlaying annotation + RNA-seq + Fur binding sites in MetaScope** and reading the regulation integratively (`module4_integrative_metascope.png`)

---

Run `/log` before closing this session.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module4): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
